# GWjax — Full Parameter Estimation on Colab GPU (Two-Phase Sampler)

This is the two-phase variant of the GWjax PE notebook. It uses
`gwjax.GWjaxTwoPhaseNestedSampler`, which splits the nested-sampling run into:

1. **Phase 1 (bulk):** a large `num_delete` per iteration, vmap-batched, that
   contracts the prior volume quickly. Runs until the live-evidence
   contribution drops below `phase1_delta_logz_threshold`.
2. **Phase 2 (tail):** classical Skilling NS with `num_delete = 1` from the
   same `NSState`, for an unbiased final evidence integral and tight posterior.

The pipeline mirrors `gwjax_colab_pe.ipynb` step-for-step until the sampler
section, where the single-phase `run` is replaced by `run_two_phase`. Two
JIT compilations occur (one per phase, because `num_delete` is baked into
the XLA program); a progress message is printed for each.

**Runtime → Change runtime type → T4 GPU** before running. Total wall-clock
is a few minutes on a T4.

## 1. Install GWjax

GWjax currently lives in a **private** GitHub repo, so `pip` needs a Personal Access Token (PAT) to clone it. Two ways to provide one:

- **(Recommended) Colab Secrets** — click the key icon 🔑 in the left sidebar → *Add new secret* → name it `GH_TOKEN`, paste your PAT, and toggle *Notebook access* on. The next cell will pick it up automatically and you never see a prompt.
- **One-off prompt** — if no Colab secret is set, the cell falls back to `getpass.getpass()` so you can paste the token without it appearing in the notebook output.

Generate a PAT at <https://github.com/settings/tokens?type=beta>. A **fine-grained** token scoped to `saulo-albuquerque-phys/GWjax` with **Contents: read-only** is enough.

In [ ]:
import os, getpass

OWNER, REPO = "saulo-albuquerque-phys", "GWjax"

# ── 0. Pin JAX to the version Colab's CUDA plugin still understands ─────
# Colab's GPU runtimes currently ship jax_cuda12_plugin 0.5.x, which calls
# `register_custom_type_id_handler` — that API was removed in jaxlib 0.10+.
# Pip will happily upgrade jax/jaxlib past that point and break CUDA, so
# we explicitly downgrade to the 0.4.x line that both the plugin and the
# pinned handley-lab/blackjax fork were built against.
!pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
!pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

# ── 1. Auth: GitHub PAT (Colab Secrets first, getpass fallback) ─────────
GH_TOKEN = None
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    pass
if not GH_TOKEN:
    GH_TOKEN = getpass.getpass(f"GitHub PAT (for {OWNER}/{REPO}): ")
os.environ["GH_TOKEN"] = GH_TOKEN

# ── 2. Install gwjax (and corner for the plot) ──────────────────────────
!pip install -q "gwjax[data] @ git+https://$GH_TOKEN@github.com/{OWNER}/{REPO}.git"
!pip install -q corner

# Scrub the token from environment and namespace.
del os.environ["GH_TOKEN"]
del GH_TOKEN

# ── 3. Restart-runtime reminder ─────────────────────────────────────────
# If JAX was already imported in this session before this cell ran,
# the new wheel won't be picked up until you restart the runtime
# (Runtime → Restart session). Re-run from this cell afterwards.
print("\n✓ install done. If you just downgraded jax, "
      "Runtime → Restart session, then re-run from cell 3.")

In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Enable double precision — important for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwjax
print("gwjax version :", gwjax.__version__)
print("JAX devices   :", jax.devices())

## 2. Build a detector network

We use a 4-second segment at 2048 Hz, analysing the 20–512 Hz band. The `TimeFrequencyGrid` is shared across all detectors in the network.

In [ ]:
DURATION       = 4.0
SAMPLING_RATE  = 2048.0
F_MIN, F_MAX   = 20.0, 512.0

grid = gwjax.TimeFrequencyGrid(
    duration=DURATION, sampling_rate=SAMPLING_RATE,
    f_min=F_MIN, f_max=F_MAX,
)
network = gwjax.Network.from_names(["H1", "L1"], grid)
print(grid)
print(network)

## 3a. Synthetic injection (default — fast, deterministic)

Generate coloured Gaussian noise at the aLIGO design PSD, then inject a GW150914-like IMRPhenomD signal. The network optimal SNR ends up around 19.

In [ ]:
TRUE_PARAMS = dict(
    m1=35.0, m2=30.0, chi_1=0.0, chi_2=0.0,
    distance=410.0, inclination=0.4,
    tc=0.0, phi_c=0.0,
    ra=1.375, dec=-1.21, psi=0.0,
)

network.generate_noise(seed=0)
waveform_fn = gwjax.build_ripplegw_waveform_fn("IMRPhenomD", f_ref=20.0)

hp, hc = waveform_fn(TRUE_PARAMS, grid.frequency_domain_array)
h_dict = network.project_waveform(
    hp, hc, TRUE_PARAMS["ra"], TRUE_PARAMS["dec"], TRUE_PARAMS["psi"], gmst=0.0,
)
for name, snr in network.optimal_snr(h_dict).items():
    print(f"  injected {name} SNR = {float(snr):.1f}")
print(f"  network SNR = {float(network.network_optimal_snr(h_dict)):.1f}")

network.inject_signal(h_dict, domain="fd")

## 3b. (Optional) Real GW150914 strain from GWOSC

Uncomment and run this cell **instead of 3a** to PE the real event. The `attach_event_to_network` helper:

- downloads the H1 and L1 strain around the GW150914 trigger (`gwpy.TimeSeries.fetch_open_data`),
- estimates each IFO's PSD from an adjacent off-source segment via Welch,
- crops + resamples the on-source segment, and attaches everything to the network.

If you ran cell 3a, **reset the network** first by re-running cell 2.

In [ ]:
# Uncomment to use real data instead of the synthetic injection:
#
# # GW150914 standard analysis: 4 s @ 4 kHz, 20–1024 Hz.
# grid = gwjax.TimeFrequencyGrid(
#     duration=4.0, sampling_rate=4096.0, f_min=20.0, f_max=1024.0,
# )
# network = gwjax.Network.from_names(["H1", "L1"], grid)
# gwjax.compat.attach_event_to_network(
#     network, "GW150914",
#     estimate_psd=True, psd_segment_duration=32.0, psd_offset=8.0,
# )
# # The merger lands at 0.875 * duration into the segment — center the tc
# # prior on that value (see PARAM_BOUNDS in cell 11).
# TRUE_PARAMS = None

## 4. Set up the two-phase nested sampler

Same 8-dimensional uniform prior over component masses, distance, inclination, and the four sky/time parameters. The only thing that changes from `gwjax_colab_pe.ipynb` is the *sampler class* — everything else (priors, fixed params, waveform) is identical.

In [ ]:
# tc convention.
# - Synthetic injection (cell 7): `inject_signal` puts the merger at
#   t = TRUE_PARAMS["tc"] = 0 in the segment — so the tc prior is a small
#   ±50 ms window around 0.
# - Real GWOSC data via `attach_event_to_network` (cell 9): the data is
#   cropped so the merger lands at 0.875 × grid.duration into the segment
#   (e.g. 3.5 s for a 4-s segment). The tc prior must be centred there.
if TRUE_PARAMS is not None:
    TC_CENTER, TC_HALFWIDTH = 0.0, 0.05
else:
    TC_CENTER, TC_HALFWIDTH = 0.875 * grid.duration, 0.5

# Sample (m1, m2) independently. IMRPhenomD is symmetric under
# (m1, m2) → (m2, m1) so the posterior has two equally-likely label-swap
# modes. We collapse them in cell 15 with a JIT-traceable
# heavier-first relabelling — no information is lost, and the
# marginal medians snap to the labelled truth (m1 = heavier, m2 = lighter).
PARAM_BOUNDS = {
    "m1":          (10.0, 80.0),
    "m2":          (10.0, 80.0),
    "distance":    (50.0, 1500.0),
    "inclination": (0.0,  float(jnp.pi)),
    "ra":          (0.0,  2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0,  float(jnp.pi)),
    "phi_c":       (0.0,  2.0 * float(jnp.pi)),
    "tc":          (TC_CENTER - TC_HALFWIDTH, TC_CENTER + TC_HALFWIDTH),
}
FIXED_PARAMS = {"chi_1": 0.0, "chi_2": 0.0}

sampler = gwjax.GWjaxTwoPhaseNestedSampler(
    network       = network,
    waveform_fn   = waveform_fn,
    param_bounds  = PARAM_BOUNDS,
    fixed_params  = FIXED_PARAMS,
    gmst          = 0.0,
)
print(f"sampling dimension: {len(sampler.param_bounds)}")
print(f"tc prior centred at {TC_CENTER:.3f} s   "
      f"({'synthetic' if TRUE_PARAMS is not None else 'real-data'} convention)")

## 5. Run the two-phase sampler

Tuning notes:

- `phase1_num_delete` should sit between `num_live // 100` (very safe, modest speedup) and `num_live // 20` (faster, slight bias from the larger batch). Default below is `num_live // 20 = 25`.
- `phase1_delta_logz_threshold` controls when to switch to phase 2. `-1.0` means *the live points still hold ≈ 37 % of total `Z`*; `-2.0` means ≈ 14 %; `0.0` switches as soon as the live evidence starts shrinking.
- `phase2_num_delete = 1` is the classical Skilling choice (unbiased).
- `log_dlogz_target = -3.0` is the final convergence tolerance for phase 2.

**Compilation warning.** Each phase compiles a separate XLA program (the `num_delete` is baked in), so you will see two "JIT-compiling … kernel" messages — typically 30–90 s each on a CPU runtime, much faster on a T4 GPU. Steady-state iterations after that are *fast*.

In [ ]:
import time

NUM_LIVE        = 500
NUM_INNER_STEPS = 40    # ~5*d for d=8

t0 = time.perf_counter()
result = sampler.run_two_phase(
    rng_key                      = jax.random.PRNGKey(0),
    num_live                     = NUM_LIVE,
    num_inner_steps              = NUM_INNER_STEPS,

    # Phase 1 (bulk, batched delete)
    phase1_num_delete            = max(1, NUM_LIVE // 20),   # = 25 here
    phase1_delta_logz_threshold  = -1.0,
    phase1_max_iterations        = 1000,

    # Phase 2 (Skilling tail, classical)
    phase2_num_delete            = 1,
    phase2_max_iterations        = 5000,
    log_dlogz_target             = -3.0,

    num_posterior_samples        = 2000,
    verbose                      = True,
)
elapsed = time.perf_counter() - t0

print(f"\ntotal wall-clock         = {elapsed:.1f} s")
print(f"  phase-1 iterations     = {result.phase1_iterations}")
print(f"  phase-2 iterations     = {result.phase2_iterations}")
print(f"  log Z                  = {result.logZ:+.3f} ± {result.logZ_err:.3f}")
print(f"  ESS                    = {result.ess:.1f}")

## 6. Posterior summary

If you see ⚠ collapsed columns / very low ESS below, the NS budget was too
small for the dimensionality. Bump `NUM_LIVE` to 800–1000 or `NUM_INNER_STEPS`
to 50 and re-run cells 11 → 13.

In [ ]:
# Heavier-first relabelling.
# IMRPhenomD is symmetric under (m1, m2) → (m2, m1), so the raw posterior
# has two equally-likely label-swap modes. The element-wise max/min
# collapse maps every sample to the heavier-first convention — single,
# unambiguous, information-preserving. `jnp.maximum`/`jnp.minimum` are
# vectorised over the sample axis and JAX-JIT-compatible (this could be
# wrapped in `@jax.jit` if it ever needed to live inside a hot loop;
# for a one-shot post-processing step the bare ops are already optimal).
_m1, _m2 = result.posterior_samples["m1"], result.posterior_samples["m2"]
result.posterior_samples = dict(result.posterior_samples)
result.posterior_samples["m1"] = jnp.maximum(_m1, _m2)
result.posterior_samples["m2"] = jnp.minimum(_m1, _m2)

print(f"  {'param':12s}  {'median':>10s}  {'-1σ':>8s}  {'+1σ':>8s}  truth")
degenerate = []
for name in sampler.param_bounds:
    s = jnp.asarray(result.posterior_samples[name])
    lo, mid, hi = jnp.percentile(s, jnp.array([16.0, 50.0, 84.0]))
    if (hi - lo) < 1e-12 * jnp.maximum(jnp.abs(mid), 1.0):
        degenerate.append(name)
    truth_str = f"{TRUE_PARAMS[name]:+.3f}" if TRUE_PARAMS is not None else "—"
    print(f"  {name:12s}  {float(mid):+10.3f}  {float(mid-lo):8.3f}  {float(hi-mid):8.3f}  {truth_str}")

if degenerate:
    print(
        f"\n⚠  posterior columns collapsed (no spread): {degenerate}\n"
        f"   ESS={result.ess:.1f}  →  NS likely didn't converge.\n"
        f"   Bump NUM_LIVE to 800 and NUM_INNER_STEPS to 50, then re-run cell 13."
    )

## 7. Corner plot

The plot range is forced to the prior bounds via `range=`, so the plot always
renders even when the posterior is degenerate — but in that case the
histograms will look like spikes (use the summary table above to diagnose).

In [ ]:
import corner

names  = list(sampler.param_bounds.keys())
data   = np.column_stack([np.asarray(result.posterior_samples[n]) for n in names])
truths = [TRUE_PARAMS[n] for n in names] if TRUE_PARAMS is not None else None
ranges = [sampler.param_bounds[n] for n in names]

fig = corner.corner(
    data, labels=names, truths=truths,
    range=ranges,
    quantiles=[0.16, 0.5, 0.84], show_titles=True,
    title_kwargs={"fontsize": 10},
)
fig.set_size_inches(11, 11)
plt.show()

## What next?

- **Tune `phase1_num_delete`**: increase toward `num_live // 10` for more wall-clock speedup; decrease toward `num_live // 100` to minimise bias from the batched-delete approximation.
- **Tune `phase1_delta_logz_threshold`**: more negative (e.g. `-2.0` or `-3.0`) means *spend more time in the cheap regime*; closer to `0` means *transition early and rely on phase 2 for the bulk*.
- **Use your own waveform**: wrap any pure-JAX `hp, hc = wf(freqs, **params)` in `gwjax.CustomWaveform` and pass it as `waveform_fn=` (no need for ripplegw).
- **Add more detectors**: include `"V1"`, `"K1"`, or even `"ET"`/`"CE"` in the `Network.from_names` list.
- **Sample spins**: move `chi_1`, `chi_2` from `FIXED_PARAMS` to `PARAM_BOUNDS` (e.g. each in `(-0.9, 0.9)`).
- **Run locally**: same workflow as a CLI script —
  ```bash
  python examples/run_pe_local.py --two-phase \
      --num-live 500 --num-inner-steps 40 \
      --phase1-num-delete 25 --phase1-delta-logz-threshold -1.0 \
      --phase2-num-delete 1 --max-iterations 5000 --log-dlogz-target -3.0
  ```
- **Single-phase variant**: the canonical reference is
  [`examples/gwjax_colab_pe.ipynb`](https://github.com/saulo-albuquerque-phys/GWjax/blob/main/examples/gwjax_colab_pe.ipynb).